# Figure 2

In [1]:
import numpy as np
import pickle
from pathlib import Path

# plot related
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

plt.rcParams["svg.fonttype"] = "none"

# model related
from nntp.utils.datatype import mean_batch_metrics, transpose_metric_history
from nntp.utils.plot import (
    reorder_legend_handles_row_major,
    find_experiments,
    load_experiments_parallel,
)

from common import (
    subtitle_fontsize,
    panel_indexing_fontsize,
    feature_size,
    output_size,
    CHANNEL,
    CHANNEL_NAME_MAPPING,
    COLORS,
)

In [2]:
def get_experiment_tasks(name):
    if name == "frdmmd":
        return [
            "fdgo-ry",
            "fdanti-ry",
            "reactgo-ry",
            "reactanti-ry",
            "delaygo-ry",
            "delayanti-ry",
            "multidm-ry",
            "multidelaydm-ry",
        ]
    elif name == "mdmdrf":
        return [
            "multidelaydm-ry",
            "multidm-ry",
            "delaygo-ry",
            "delayanti-ry",
            "reactgo-ry",
            "reactanti-ry",
            "fdgo-ry",
            "fdanti-ry",
        ]


# create path
out_path = Path("./output")
out_path.mkdir(parents=True, exist_ok=True)
root_path = Path("..").expanduser().resolve() / "figures/experiments"
experiments = find_experiments(root_path, get_experiment_tasks)
for path in experiments:
    print(path)


def build_load_experiment(name):
    def load_experiment(_, path):
        return pickle.load(open(path / name, "rb"))

    return load_experiment


loaded_experiments_train = load_experiments_parallel(
    experiments, build_load_experiment("metadata_train_mean_metrics.blob")
)
loaded_experiments_validation = load_experiments_parallel(
    experiments, build_load_experiment("metadata_validation_mean_metrics.blob")
)

{'tasks': ['fdgo-ry', 'fdanti-ry', 'reactgo-ry', 'reactanti-ry', 'delaygo-ry', 'delayanti-ry', 'multidm-ry', 'multidelaydm-ry'], 'paths': [PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/experiments/frdmmd/25035270/1/72dc7938e5066ece'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/experiments/frdmmd/25035270/42/01737adfea12e65c'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/experiments/frdmmd/25035270/128/fabc6213f847706f'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/experiments/frdmmd/25035270/128128/d89bcd5943f241a8')]}
{'tasks': ['multidelaydm-ry', 'multidm-ry', 'delaygo-ry', 'delayanti-ry', 'reactgo-ry', 'reactanti-ry', 'fdgo-ry', 'fdanti-ry'], 'paths': [PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/experiments/mdmdrf/25035271/1/72dc7938e5066ece'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/figures/experiments/mdmdrf/25035271/42/01737adfea12e65c'), PosixPath('/allen/aind/scratch/ivan.y.gao/CoSyn-RNN/

In [3]:
def plot_experiments(
    ax,
    experiments,
    metric,
    title,
    logY=False,
    show_xlabel=False,
    show_ylabel=False,
):
    tasks = experiments["tasks"]
    data = experiments["data"]

    # =====================================================
    # Plot model-level metrics
    # =====================================================
    model_metric_keys = {
        "sparsity": "loss/sparsity",
        "amplification": "penalty_amplification",
    }

    if metric in model_metric_keys:
        key = model_metric_keys[metric]

        curves = np.stack(
            [np.asarray(seed_data["model"][key], dtype=float) for seed_data in data],
            axis=0,
        )

        mean_curve = np.nanmean(curves, axis=0)
        lower = np.nanmin(curves, axis=0)
        upper = np.nanmax(curves, axis=0)
        x = np.arange(mean_curve.size)

        ax.fill_between(
            x,
            lower,
            upper,
            color="gray",
            alpha=0.4,
            linewidth=0,
        )

        ax.plot(
            x,
            mean_curve,
            color="gray",
            linewidth=1,
        )

        if metric == "amplification":
            ax.axhline(
                y=1.0,
                xmin=0.05,
                xmax=0.95,
                color="black",
                linestyle="--",
                linewidth=0.8,
                alpha=0.7,
                zorder=1,
            )

    # =====================================================
    # Plot task-level metrics
    # =====================================================
    else:
        for task in tasks:
            if metric == "train":
                curves = np.stack(
                    [
                        np.asarray(
                            seed_data["task"]["loss"][task]
                            - seed_data["model"]["loss/sparsity"],
                            dtype=float,
                        )
                        for seed_data in data
                    ],
                    axis=0,
                )

            elif metric == "validation":
                curves = np.stack(
                    [
                        np.asarray(
                            seed_data["task"]["loss"][task],
                            dtype=float,
                        )
                        for seed_data in data
                    ],
                    axis=0,
                )

            elif metric == "accuracy":
                curves = np.stack(
                    [
                        np.asarray(
                            seed_data["task"]["accuracy_angle"][task],
                            dtype=float,
                        )
                        for seed_data in data
                    ],
                    axis=0,
                )

            else:
                raise ValueError(f"Unsupported metric: {metric}")

            mean_curve = np.nanmean(curves, axis=0)
            lower = np.nanmin(curves, axis=0)
            upper = np.nanmax(curves, axis=0)
            x = np.arange(mean_curve.size)

            color = COLORS[task]

            ax.fill_between(
                x,
                lower,
                upper,
                color=color,
                alpha=0.4,
                linewidth=0,
            )

            ax.plot(
                x,
                mean_curve,
                color=color,
                linewidth=1,
                label=task.removesuffix("-ry"),
            )

    # =====================================================
    # Metric-specific formatting
    # =====================================================
    if metric == "accuracy":
        chance = (2 * 36) / 360

        ax.axhline(
            y=chance,
            xmin=0.05,
            xmax=0.95,
            color="black",
            linestyle="--",
            linewidth=0.8,
            alpha=0.7,
            zorder=1,
        )

        ax.text(
            0.94,
            chance + 0.02,
            "Chance",
            transform=ax.get_yaxis_transform(),
            ha="right",
            va="bottom",
            fontsize=10,
            color="black",
        )

        ax.set_ylim(0, 1)
        ax.set_yticks([0, 1])
        ax.tick_params(
            axis="y",
            which="major",
            length=0,
        )

    if logY:
        ax.set_yscale("log")

    # =====================================================
    # Axis labels and ticks
    # =====================================================
    if show_xlabel:
        ax.set_xlabel("Epoch")
        ax.tick_params(
            axis="x",
            which="both",
            bottom=True,
            labelbottom=True,
        )
    else:
        ax.tick_params(
            axis="x",
            which="both",
            bottom=False,
            labelbottom=False,
        )

    if show_ylabel:
        if metric in {"train", "validation", "sparsity"}:
            ylabel = "Loss"
        elif metric == "accuracy":
            ylabel = "Accuracy"
        else:
            ylabel = metric.replace("_", " ").capitalize()

        if logY:
            ylabel += " (log)"

        ax.set_ylabel(ylabel)
        ax.tick_params(
            axis="y",
            which="both",
            left=True,
            labelleft=True,
        )
        ax.tick_params(
            axis="y",
            which="minor",
            left=False,
            labelleft=False,
        )
    else:
        ax.set_ylabel("")
        ax.tick_params(
            axis="y",
            which="both",
            left=False,
            labelleft=False,
        )

    # =====================================================
    # General formatting
    # =====================================================
    ax.set_title(title)

    sns.despine(
        ax=ax,
        top=True,
        right=True,
        left=not show_ylabel,
        bottom=not show_xlabel,
    )

    ax.grid(
        axis="x",
        which="major",
        linestyle="--",
        linewidth=0.6,
        alpha=0.5,
    )

    return ax

In [4]:
fig, axes = plt.subplots(6, 2, figsize=(16, 8), sharex=True, sharey="row")

plot_experiments(
    axes[0, 0],
    loaded_experiments_train[0],
    metric="train",
    title="Training Task Loss",
    logY=True,
    show_ylabel=True,
)
plot_experiments(
    axes[1, 0],
    loaded_experiments_train[0],
    metric="accuracy",
    title="Training Accuracy",
    show_ylabel=True,
)
plot_experiments(
    axes[2, 0],
    loaded_experiments_validation[0],
    metric="validation",
    title="Validation Loss",
    logY=True,
    show_ylabel=True,
)
plot_experiments(
    axes[3, 0],
    loaded_experiments_validation[0],
    metric="accuracy",
    title="Validation Accuracy",
    show_ylabel=True,
)
plot_experiments(
    axes[4, 0],
    loaded_experiments_train[0],
    metric="sparsity",
    title="Training Sparsity Loss",
    logY=True,
    show_ylabel=True,
)
plot_experiments(
    axes[5, 0],
    loaded_experiments_train[0],
    metric="amplification",
    title="Loss-Dependent Amplification Factor",
    show_xlabel=True,
    show_ylabel=True,
)
plot_experiments(
    axes[0, 1],
    loaded_experiments_train[1],
    metric="train",
    title="Training Task Loss",
    logY=True,
)
plot_experiments(
    axes[1, 1],
    loaded_experiments_train[1],
    metric="accuracy",
    title="Training Accuracy",
)
plot_experiments(
    axes[2, 1],
    loaded_experiments_validation[1],
    metric="validation",
    title="Validation Loss",
    logY=True,
)
plot_experiments(
    axes[3, 1],
    loaded_experiments_validation[1],
    metric="accuracy",
    title="Validation Accuracy",
)
plot_experiments(
    axes[4, 1],
    loaded_experiments_train[1],
    metric="sparsity",
    title="Training Sparsity Loss",
    logY=True,
)
plot_experiments(
    axes[5, 1],
    loaded_experiments_train[1],
    metric="amplification",
    title="Loss-Dependent Amplification Factor",
    show_xlabel=True,
)

forward_legend = fig.legend(
    handles=reorder_legend_handles_row_major(
        [
            Line2D(
                [0],
                [0],
                color=COLORS[task],
                label=CHANNEL_NAME_MAPPING[task.removesuffix("-ry")],
            )
            for task in loaded_experiments_train[0]["tasks"]
        ],
        ncol=4,
    ),
    loc="upper center",
    bbox_to_anchor=(0.30, 0.98),
    ncol=4,
    frameon=False,
)
reverse_legend = fig.legend(
    handles=reorder_legend_handles_row_major(
        [
            Line2D(
                [0],
                [0],
                color=COLORS[task],
                label=CHANNEL_NAME_MAPPING[task.removesuffix("-ry")],
            )
            for task in loaded_experiments_train[1]["tasks"]
        ],
        ncol=4,
    ),
    loc="upper center",
    bbox_to_anchor=(0.70, 0.98),
    ncol=4,
    frameon=False,
)
fig.text(
    0.30,
    0.99,
    "Forward Order",
    ha="center",
    va="center",
    fontsize=subtitle_fontsize,
    fontweight="bold",
)
fig.text(
    0.70,
    0.99,
    "Reverse Order",
    ha="center",
    va="center",
    fontsize=subtitle_fontsize,
    fontweight="bold",
)
fig.subplots_adjust(wspace=0.0, hspace=0.3)
fig.align_ylabels(axes)

for label, ax in zip("abcdefghijkl", axes.flat):
    ax.text(
        0.01,
        1.05,
        f"{label}",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=panel_indexing_fontsize,
        fontweight="bold",
        clip_on=False,
    )

fig.savefig(
    out_path / "Figure2.png",
    dpi=300,
    bbox_inches="tight",
)
# fig.savefig(
#     out_path / "Figure2.svg",
#     bbox_inches="tight",
# )
plt.close(fig)

/tmp/ipykernel_279063/3821846544.py:107: RuntimeWarning: Mean of empty slice
  mean_curve = np.nanmean(curves, axis=0)
/tmp/ipykernel_279063/3821846544.py:108: RuntimeWarning: All-NaN slice encountered
  lower = np.nanmin(curves, axis=0)
/tmp/ipykernel_279063/3821846544.py:109: RuntimeWarning: All-NaN slice encountered
  upper = np.nanmax(curves, axis=0)
